# 08 · Module und Pakete

**Lernziele**

- Du weißt, was Module und Pakete sind und wie ein Projekt aufgebaut ist.
- Du kennst die Varianten von `import` und wann du welche nimmst.
- Du verstehst, was bei `import` wirklich passiert.
- Du nutzt `if __name__ == "__main__":` für Skripte.
- Du installierst Pakete von Drittanbietern mit uv.

> Grundlage: Inden, *Einstieg in Python*, Kapitel 2 und 6.

## Module, Pakete, Bibliotheken

| Begriff | Was es ist | Beispiel |
| --- | --- | --- |
| Modul | eine `.py`-Datei | `greetings.py` |
| Paket | ein Ordner mit Modulen (und meist einer `__init__.py`) | `shop/` |
| Standardbibliothek | Module, die mit Python kommen | `math`, `random`, `pathlib`, `json` |
| Drittanbieter-Paket | von [PyPI](https://pypi.org) installiert | `requests`, `pandas` |

Für dieses Notebook liegen Beispiele im Ordner `beispiele/`:

```
beispiele/
├── greetings.py
└── shop/
    ├── __init__.py
    ├── pricing.py
    └── invoice.py
```

Damit Python den Ordner findet, fügen wir ihn zum Suchpfad hinzu. Was das bedeutet, klären wir gleich.

In [ ]:
import sys

sys.path.insert(0, "../beispiele")

## Import-Varianten

In [ ]:
import math                       # ganzes Modul, Zugriff mit math.sqrt
from random import randint        # einzelner Name direkt
from datetime import date as Date  # mit anderem Namen

print(math.sqrt(16))
print(randint(1, 6))
print(Date.today())

| Variante | Wann |
| --- | --- |
| `import math` | Standard. Beim Lesen ist klar, woher `math.sqrt` kommt. |
| `from pathlib import Path` | einzelne, oft genutzte Namen |
| `import pandas as pd` | etablierte Abkürzungen |
| `from math import *` | **nie**: holt unkontrolliert alle Namen und überschreibt eigene |

Imports stehen immer **oben** in der Datei. *Explicit is better than implicit:* Beim Lesen soll sofort klar sein, woher jeder Name kommt.

**Kurzübung 1 (Umzugskartons):** 47 Ordner kommen in Umzugskartons, in einen Karton passen 12 Ordner. Berechne die Anzahl der Kartons mit der Funktion `ceil` (aufrunden) aus dem Modul `math`: einmal nach `import math`, einmal nach `from math import ceil`. Welche Fassung nimmst du, und warum?

Untersuchen: Warum ist `47 // 12 + 1` keine gute Lösung? Teste mit 48 Ordnern.

In [ ]:
import math
from math import ceil

folders = 47
per_box = 12
print(math.ceil(folders / per_box))  # 4
print(ceil(folders / per_box))       # 4

# import math: beim Lesen ist sofort klar, woher ceil kommt; gut bei seltener Nutzung.
# from math import ceil: kürzer, lohnt sich, wenn ceil oft vorkommt.

# 48 // 12 + 1 ergibt 5, gebraucht werden aber nur 4 Kartons:
# "+ 1" stimmt nur, wenn beim Teilen ein Rest bleibt.
print(48 // 12 + 1, math.ceil(48 / per_box))

## Was bei `import` passiert

In [ ]:
import greetings

Die Ausgabe zeigt: Python hat die Datei **ausgeführt**, von oben nach unten. Im Detail passiert Folgendes:

1. Python schaut in `sys.modules`, ob das Modul schon geladen ist. Wenn ja, nimmt es das fertige Modul, fertig.
2. Sonst sucht Python eine passende Datei in den Ordnern aus `sys.path`, der Reihe nach.
3. Die Datei wird einmal ausgeführt. Alle Namen darin landen in einem **Modulobjekt**.
4. Das Modulobjekt wird an den Namen `greetings` gebunden.

Ein zweiter Import führt die Datei deshalb **nicht** erneut aus:

In [ ]:
import greetings

print("greetings" in sys.modules)

Auch ein Modul ist ein Objekt mit Attributen:

In [ ]:
print(type(greetings))
print(greetings.__file__)
print(greetings.DEFAULT_GREETING)
print(greetings.greet("Ada"))

> **Im Notebook:** Änderst du eine Moduldatei, sieht das Notebook die Änderung erst nach einem Kernel-Neustart (oder `importlib.reload(greetings)`).

### Wo sucht Python?

In [ ]:
for folder in sys.path[:4]:
    print(folder)

Ganz vorne steht der Ordner des gestarteten Skripts. Das hat eine fiese Folge:

> **Typischer Fehler:** Nenne eine eigene Datei nie wie ein Modul der Standardbibliothek, also nicht `random.py`, `math.py`, `csv.py` oder `email.py`. Sonst findet `import random` deine Datei statt der echten, und plötzlich fehlt `random.randint`.

**Kurzübung 2 (Zweiter Import):** Sage vorher, welche Ausgabe die folgende Zelle erzeugt, und begründe: Erscheint die Zeile „Modul greetings wird geladen“? Führe die Zelle danach aus.

Untersuchen: Starte den Kernel neu, führe die Zelle mit `sys.path` am Anfang des Notebooks und dann nur diese Zelle aus. Was ändert sich, und warum?

In [ ]:
import greetings as gr

print(gr is greetings)
print(gr.greet("Grace", "Moin"))

In [ ]:
# Ausgabe:
# True
# Moin Grace!
# "Modul greetings wird geladen" fehlt: greetings steht schon in sys.modules,
# Python führt die Datei kein zweites Mal aus. gr ist nur ein zweiter Name
# für dasselbe Modulobjekt, deshalb ergibt "gr is greetings" True.

# Nach einem Kernel-Neustart ist das Modul noch nicht geladen: Die Zeile
# "Modul greetings wird geladen" erscheint, danach folgt ein NameError,
# weil nur der Name gr existiert, nicht greetings.

## Pakete

Ein Paket gruppiert zusammengehörige Module. Importiert wird mit Punkten:

In [ ]:
from shop.pricing import gross_price, VAT_RATE
from shop.invoice import invoice_line

print(VAT_RATE, gross_price(100))
print(invoice_line("Kaffee", 4.20))

`shop/invoice.py` nutzt selbst `from shop.pricing import gross_price`. Module dürfen sich also gegenseitig importieren.

Ein typisches kleines Projekt sieht so aus:

```
mein_projekt/
├── pyproject.toml      # Projektbeschreibung und Abhängigkeiten (uv)
├── README.md
├── src/
│   └── mein_projekt/
│       ├── __init__.py
│       ├── models.py
│       └── reports.py
└── tests/
```

Datei- und Ordnernamen: `snake_case`, keine Leerzeichen, keine Bindestriche. `mein-projekt.py` lässt sich nicht importieren.

**Kurzübung 3 (Kassenbon aus dem Paket `shop`):** Gib für den Warenkorb `cart` (Nettopreise) mit `invoice_line` je eine Bonzeile aus und berechne die Bruttosumme mit `gross_price`. Erwartet: 19.17 €.

In [ ]:
cart = {"Kaffee": 4.20, "Tasse": 8.40, "Notizblock": 3.50}

In [ ]:
total = 0
for product, net_price in cart.items():
    print(invoice_line(product, net_price))
    total += gross_price(net_price)
print(f"{'Summe':<10} {total:>8.2f} €")

# Kürzer für die Summe: sum(gross_price(price) for price in cart.values())

## Skripte: `if __name__ == "__main__":`

Jedes Modul hat einen Namen in `__name__`. Wird die Datei **direkt gestartet**, heißt sie `"__main__"`, beim **Import** trägt sie ihren Modulnamen:

In [ ]:
print(greetings.__name__)
print(__name__)  # das Notebook selbst

Darum steht der Startcode eines Skripts in diesem Block. Er läuft beim Starten, aber nicht beim Import:

```
def main():
    print(greet("Welt"))


if __name__ == "__main__":
    main()
```

```
uv run python beispiele/greetings.py
```

**Kurzübung 4 (Modul als Skript starten):** Sage vorher, welche Zeilen erscheinen, wenn du `greetings.py` als Skript startest, und vergleiche mit der Ausgabe von `import greetings` weiter oben. Starte die Datei dann im Notebook mit `!uv run python ../beispiele/greetings.py`.

In [ ]:
!uv run python ../beispiele/greetings.py
# Ausgabe:
# Modul __main__ wird geladen
# Hallo Welt!
# Als Skript heißt das Modul "__main__", deshalb läuft der Block unter
# if __name__ == "__main__". Beim Import hieß es "greetings": nur die erste Zeile erschien.

## Pakete von Drittanbietern

Pakete von PyPI installierst du mit uv. Sie landen in der `.venv` des Projekts und werden in `pyproject.toml` eingetragen:

```
uv add requests
```

Danach funktioniert `import requests` wie bei jedem anderen Modul. Prüfe vorher kurz, ob ein Paket gepflegt und verbreitet ist (Downloads, letzte Version, Dokumentation).

## Übungen

**Übung 1 (Teams auslosen, leicht):** Teile die 10 Teilnehmenden in `names` zufällig in drei Teams auf. Finde mit `help(random)` die passende Funktion selbst.

Untersuchen: Was steht in `names`, wenn du `names = random.shuffle(names)` schreibst?

In [ ]:
import random

names = "Ada Linus Guido Grace Tim Lili Matze Peter Ken Barbara".split()

In [ ]:
random.seed(42)  # gleiche Auslosung bei jedem Lauf, nur zum Nachvollziehen
shuffled = random.sample(names, k=len(names))
teams = [shuffled[i::3] for i in range(3)]
print(teams)

# names = random.shuffle(names) macht names zu None:
# shuffle mischt die Liste selbst und liefert nichts zurück.
print(random.shuffle(names))

**Übung 2 (Pendelzeiten auswerten, leicht):** In `commute_minutes` stehen deine Fahrzeiten zur Arbeit an fünf Tagen; am Freitag gab es Stau. Berechne Mittelwert und Median zuerst selbst (Median: der mittlere Wert der sortierten Liste), dann mit `mean` und `median` aus dem Modul `statistics`. Erwartet: 44.4 und 32 Minuten. Welcher Wert beschreibt die typische Fahrt besser, und warum?

Untersuchen: Füge eine sechste Fahrt mit 30 Minuten hinzu. Welchen Median liefert dann deine eigene Rechnung, welchen `statistics.median`?

In [ ]:
commute_minutes = [32, 29, 35, 31, 95]  # Montag bis Freitag

In [ ]:
import statistics

own_mean = sum(commute_minutes) / len(commute_minutes)
ordered = sorted(commute_minutes)
own_median = ordered[len(ordered) // 2]
print(own_mean, own_median)  # 44.4 32

print(statistics.mean(commute_minutes), statistics.median(commute_minutes))  # 44.4 32

# Der Median beschreibt die typische Fahrt besser: Der eine Stau-Tag zieht den
# Mittelwert auf 44,4 Minuten, so lange hat keine einzige Fahrt gedauert.

# Untersuchen: Bei sechs Werten gibt es zwei mittlere Werte. statistics.median
# nimmt ihren Durchschnitt, die eigene Rechnung nur den oberen.
six_days = sorted(commute_minutes + [30])
print(six_days[len(six_days) // 2], statistics.median(six_days))  # 32 31.5

**Übung 3 (Eigenes Modul für Dateinamen, leicht):** Aus Meeting-Titeln sollen Dateinamen werden: `"Team Meeting März"` wird zu `"team-meeting-märz.txt"`. Lege ein Modul `file_names.py` an mit der Funktion `to_file_name(title)`: Kleinbuchstaben, Leerzeichen durch `-` ersetzt, Endung `.txt`. Importiere die Funktion im Notebook und wandle alle Titel in `titles` um.

Tipp: Eine Zelle, die mit `%%writefile file_names.py` beginnt, schreibt alles darunter in die Datei. Änderst du die Datei später, sieht das Notebook die neue Fassung erst nach einem Kernel-Neustart.

Untersuchen: Was liefert deine Funktion für `" Budget  2027 "` (Leerzeichen am Rand und doppelt)? Wie machst du sie robuster?

In [ ]:
titles = ["Team Meeting März", "Budget 2027", "Kick-off Projekt Nord"]

In [ ]:
%%writefile file_names.py
"""Dateinamen aus Titeln bilden."""


def to_file_name(title: str) -> str:
    """Macht aus einem Titel einen Dateinamen: klein, Bindestriche statt Leerzeichen."""
    return title.lower().replace(" ", "-") + ".txt"

In [ ]:
from file_names import to_file_name

for title in titles:
    print(to_file_name(title))

# Untersuchen: " Budget  2027 " wird zu "-budget--2027-.txt".
# split() ohne Argument trennt an beliebig vielen Leerzeichen und wirft die Ränder weg:
print("-".join(" Budget  2027 ".lower().split()) + ".txt")  # budget-2027.txt
# Welche Fassung nimmst du? replace ist kürzer zu lesen, split/join verträgt unsaubere Eingaben.

**Übung 4 (Modul mit Schnelltest, mittel):** Ein Kollege hat das Modul `report_tools.py` für den Monatsbericht geschrieben (Zelle unten). Importierst du `summary` im Notebook, erscheint ungefragt die Ausgabe seines Schnelltests. Ändere das Modul so, dass der Schnelltest nur beim Start als Skript läuft. Prüfe beides: Der Import im Notebook bleibt ohne Ausgabe, `!uv run python report_tools.py` zeigt den Schnelltest.

Untersuchen: Führe die Import-Zelle ein zweites Mal aus, bevor du etwas änderst. Warum erscheint der Schnelltest dann nicht mehr? Was musst du nach dem Ändern der Datei tun, damit das Notebook die neue Fassung sieht?

In [ ]:
%%writefile report_tools.py
"""Kennzahlen für den Monatsbericht."""

import statistics


def summary(amounts: list[float]) -> str:
    """Fasst Monatsumsätze in einem Satz zusammen."""
    return (
        f"{len(amounts)} Monate, Summe {sum(amounts):.2f} €, "
        f"Schnitt {statistics.mean(amounts):.2f} €, "
        f"Median {statistics.median(amounts):.2f} €"
    )


# Schnelltest
print(summary([1200, 950, 1430, 870]))

In [ ]:
from report_tools import summary

In [ ]:
%%writefile report_tools.py
"""Kennzahlen für den Monatsbericht."""

import statistics


def summary(amounts: list[float]) -> str:
    """Fasst Monatsumsätze in einem Satz zusammen."""
    return (
        f"{len(amounts)} Monate, Summe {sum(amounts):.2f} €, "
        f"Schnitt {statistics.mean(amounts):.2f} €, "
        f"Median {statistics.median(amounts):.2f} €"
    )


if __name__ == "__main__":
    print(summary([1200, 950, 1430, 870]))  # Schnelltest, nur beim Start als Skript

In [ ]:
import importlib

import report_tools

# Ein zweites "import" zeigt den Schnelltest nicht mehr: Das Modul steht schon
# in sys.modules, Python führt die Datei nicht erneut aus. Aus demselben Grund
# kennt das Notebook noch die alte Fassung. reload (oder ein Kernel-Neustart)
# lädt die geänderte Datei, jetzt ohne Ausgabe.
importlib.reload(report_tools)

print(report_tools.summary([980, 1100, 1250]))

In [ ]:
!uv run python report_tools.py
# 4 Monate, Summe 4450.00 €, Schnitt 1112.50 €, Median 1075.00 €

**Übung 5 (Passwort-Werkzeug als Modul, mittel):** Lege `password_tools.py` an mit `password_problems` aus 07 und `generate_password(length=16)`, die mit `secrets` ein Passwort erzeugt. Ein `__main__`-Block gibt ein neues Passwort und seine Prüfung aus. Importiere die Funktionen im Notebook (keine Ausgabe), starte die Datei dann mit `!uv run python password_tools.py` als Skript.

Tipp: Eine Zelle, die mit `%%writefile password_tools.py` beginnt, schreibt alles darunter in die Datei.

In [ ]:
%%writefile password_tools.py
"""Passwörter prüfen und erzeugen."""

import secrets
import string

ALPHABET = string.ascii_letters + string.digits + "!?#%&"


def password_problems(password: str, min_length: int = 12) -> list[str]:
    """Liefert die Mängel eines Passworts; leere Liste heißt sicher."""
    problems = []
    if len(password) < min_length:
        problems.append(f"kürzer als {min_length} Zeichen")
    if not any(char.isdigit() for char in password):
        problems.append("Ziffer fehlt")
    if not any(char.isupper() for char in password):
        problems.append("Großbuchstabe fehlt")
    if not any(char.islower() for char in password):
        problems.append("Kleinbuchstabe fehlt")
    return problems


def generate_password(length: int = 16) -> str:
    """Erzeugt ein zufälliges Passwort; secrets statt random, weil sicher."""
    return "".join(secrets.choice(ALPHABET) for _ in range(length))


if __name__ == "__main__":
    password = generate_password()
    print(password, password_problems(password) or "sicher")

In [ ]:
from password_tools import generate_password, password_problems  # keine Ausgabe

In [ ]:
!uv run python password_tools.py

## Zusammenfassung

- Modul = `.py`-Datei, Paket = Ordner mit Modulen.
- Bevorzugt `import modul` oder `from modul import name`, nie `import *`.
- `import` sucht in `sys.path`, führt die Datei **einmal** aus und merkt sie sich in `sys.modules`.
- Eigene Dateien nie wie Standardmodule nennen.
- Startcode in `if __name__ == "__main__":`.
- Drittanbieter-Pakete: `uv add paketname`.

> **Weiter im Buch:** Als Nächstes käme Kapitel 7 (Dateien). Wir ziehen Kapitel 9 (Fehlerbehandlung) vor, weil wir `try` und `with` beim Arbeiten mit Dateien brauchen.